# Mestrado em Computação Aplicada • ConvNeXt-Tiny (Otimizado + Segmentação YOLO-World & SAM 2)
**Autor:** José Vinícius da Silva de Souza · **Coautor & Orientador:** Prof. Dr. Rodrigo Sant'Ana  
**Instituição:** UNIVALI — Universidade do Vale do Itajaí · **LEMA** (Laboratório de Estudos Marinhos Aplicados)

Classificação de imagens de grandes peixes migratórios — **mesmos splits da execução de 11/10/2026** (`run_20261011_165821`: 4.354 treino / 933 validação / 933 teste) para comparação científica justa no artigo/dissertação.

### Melhorias aplicadas em relação ao baseline (72%)
1. **Segmentação e recorte automático (`USAR_DATASET_SEGMENTADO = True`)**: usa os recortes gerados por `segmentar_peixes_auto.py` (YOLO-World + SAM 2.1), removendo viés de fundo (convés, pescadores, água) e preservando os mesmos splits do baseline.
2. **Preservação de proporção (`resize_with_pad`)**: evita achatar o corpo e o rostro/nadadeiras de peixes alongados (ex.: *Tetrapturus pfluegeri*, marlins, atuns).
3. **Resolução maior (`320×320`, ajustável para `384×384` na A100)**: preserva detalhes taxonômicos finos (nadadeira dorsal, peitoral, rostro e linha lateral).
4. **Substituição da `Lambda` por `layers.Rescaling(255.0)`**: gera um `.keras` 100% compatível com `keras.models.load_model()` (pronto para colocar em `backend/models/modelo.keras` no **FishID Studio**).
5. **Fine-tuning com Warmup + CosineDecay + `AdamW` (`weight_decay=1e-2`, `clipnorm=1.0`)**: evita queda precoce do *learning rate* e carrega o melhor checkpoint da Fase A antes de iniciar a Fase B.
6. **Data Augmentation de campo + MixUp leve (`alpha=0.2`) + Cache em SSD local do Colab**: treino até 5× mais rápido evitando gargalo do Google Drive FUSE.
7. **Test-Time Augmentation (TTA)** + **Grad-CAM comparativo**: reporta métricas lado a lado com o baseline original.


## 0. Auto-Segmentação Turbo do Dataset (YOLO-World + SAM 2.1)

Roda a segmentação completa (`crops/`, `masks/` e `split_*_cropped.csv`) direto no Google Drive com:
- **União YOLO-World (`yolov8l-worldv2.pt`) + SAM 2.1 (`sam2.1_t.pt`) + margem de 15%**: nunca corta o bico/rostro fino de espadartes e marlins.
- **Filtro anti-placa-de-fundo**: descarta micro-detecções (< 4% quando há peixe maior; < 2,5% usa fallback seguro).
- **Retomada automática (cache)** + **Staging em SSD local do Colab**: se desconectar no meio, basta rodar de novo que ele continua de onde parou.
- Depois que todas as ~6.220 fotos já estiverem segmentadas no Drive, basta mudar `RODAR_SEGMENTACAO = False`.


In [ ]:
from __future__ import annotations

# =============================================================================
# 0. Auto-Segmentação Turbo (YOLO-World + SAM 2.1) — dataset completo no Drive
# =============================================================================
RODAR_SEGMENTACAO = False      # Mude para True apenas se precisar re-segmentar o dataset
LIMITE_SEGMENTACAO = None      # None = todas as ~6.220 fotos (~10–18 min na T4) | 20 = teste rápido
LIMPAR_TESTE_ANTIGO = True     # Limpa SOMENTE se houver <= 100 fotos (teste antigo de 20); nunca apaga o dataset inteiro!

if RODAR_SEGMENTACAO:
    import importlib.util, subprocess, sys
    if importlib.util.find_spec("ultralytics") is None:
        print("Instalando ultralytics...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "opencv-python-headless", "tqdm"])

    try:
        from google.colab import drive
        if not __import__("pathlib").Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except ImportError:
        pass


    import argparse
    import os
    import shutil
    import time
    from concurrent.futures import ThreadPoolExecutor
    from pathlib import Path

    import cv2
    import numpy as np
    import pandas as pd
    from tqdm import tqdm

    # =============================================================================
    # Configurações padrão (funcionam via terminal OU colando direto numa célula do Colab!)
    # =============================================================================
    DEFAULT_SPLITS_DIR: str = os.getenv(
        "FISHID_SPLITS_DIR",
        "/content/drive/MyDrive/modelos_peixes/fonte_verdade_experimentos/run_20261011_165821",
    )
    DEFAULT_OUT_DIR: str = os.getenv(
        "FISHID_SEG_OUT_DIR",
        "/content/drive/MyDrive/modelos_peixes/dataset_segmentado",
    )
    DEFAULT_LIMIT: int | None = None      # None = dataset completo (~6.220 fotos); coloque 20 só para teste rápido
    DEFAULT_USE_SAM: bool = True          # Mude para False se quiser apenas os recortes ultrarrápidos (sem máscara)
    DEFAULT_BATCH_SIZE: int = 16          # Lote de imagens por chamada do YOLO-World na GPU
    DEFAULT_MAX_INFER_SIDE: int = 1280    # Reduz fotos 4K/12MP apenas na inferência (recorte final mantém resolução original)
    DEFAULT_IO_WORKERS: int = 8           # Threads paralelas para leitura/escrita

    # Classes textuais open-vocabulary otimizadas para grandes peixes migratórios (alto recall em fotos de campo)
    FISH_PROMPTS: list[str] = [
        "fish",
        "large fish",
        "ocean fish",
        "tuna",
        "marlin",
        "sailfish",
        "swordfish",
        "billfish",
        "mahi mahi",
        "dolphin fish",
        "shark",
        "dead fish",
        "caught fish",
        "seafood",
    ]


    def _clamp_box_with_margin(
        x1: float,
        y1: float,
        x2: float,
        y2: float,
        img_w: int,
        img_h: int,
        margin_ratio: float,
    ) -> tuple[int, int, int, int]:
        bw = max(1.0, x2 - x1)
        bh = max(1.0, y2 - y1)
        mx = bw * margin_ratio
        my = bh * margin_ratio
        nx1 = max(0, int(round(x1 - mx)))
        ny1 = max(0, int(round(y1 - my)))
        nx2 = min(img_w, int(round(x2 + mx)))
        ny2 = min(img_h, int(round(y2 + my)))
        return nx1, ny1, max(nx1 + 1, nx2), max(ny1 + 1, ny2)


    def _select_best_box(
        boxes_xyxy: np.ndarray,
        confs: np.ndarray,
        img_area: float,
        min_primary_area_frac: float = 0.04,
    ) -> tuple[np.ndarray, float, float]:
        """Escolhe o peixe principal ponderando confiança e área relativa (ignora placas/ruídos minúsculos)."""
        areas_frac = [
            min(1.0, max(1.0, float((b[2] - b[0]) * (b[3] - b[1]))) / max(img_area, 1.0))
            for b in boxes_xyxy
        ]
        # Se houver pelo menos uma detecção razoável (>= 4% da imagem), descarta micro-caixas do fundo
        valid_idxs = [i for i, af in enumerate(areas_frac) if af >= min_primary_area_frac]
        candidate_idxs = valid_idxs if valid_idxs else list(range(len(boxes_xyxy)))

        best_idx = candidate_idxs[0]
        best_score = -1.0
        for idx in candidate_idxs:
            af = areas_frac[idx]
            score = float(confs[idx]) * (0.45 + 0.55 * np.sqrt(af))
            if score > best_score:
                best_score = score
                best_idx = idx
        return boxes_xyxy[best_idx], float(confs[best_idx]), float(areas_frac[best_idx])


    def _mask_to_clean_bbox(mask_u8: np.ndarray) -> tuple[int, int, int, int] | None:
        ys, xs = np.where(mask_u8 > 0)
        if len(xs) == 0 or len(ys) == 0:
            return None
        return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


    def _resize_for_inference(img_bgr: np.ndarray, max_side: int) -> tuple[np.ndarray, float]:
        """Redimensiona imagens muito grandes para acelerar YOLO/SAM e retorna o fator de escala."""
        h, w = img_bgr.shape[:2]
        longest = max(h, w)
        if max_side <= 0 or longest <= max_side:
            return img_bgr, 1.0
        scale = max_side / float(longest)
        new_w = max(1, int(round(w * scale)))
        new_h = max(1, int(round(h * scale)))
        resized = cv2.resize(img_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
        return resized, scale


    def _read_image_task(item: tuple[int, Path, int]) -> tuple[int, np.ndarray | None, np.ndarray | None, float]:
        """Lê imagem do disco e já prepara a versão redimensionada para inferência (em thread paralela)."""
        idx, path, max_side = item
        img_bgr = cv2.imread(str(path))
        if img_bgr is None:
            return idx, None, None, 1.0
        infer_bgr, scale = _resize_for_inference(img_bgr, max_side)
        return idx, img_bgr, infer_bgr, scale


    def _write_image_task(args: tuple[Path, np.ndarray, list[int]]) -> None:
        path, img, params = args
        cv2.imwrite(str(path), img, params)


    def _sync_directory_parallel(src_root: Path, dst_root: Path, workers: int = 8) -> int:
        """Copia arquivos do SSD local para o Google Drive em paralelo (muito mais rápido que FUSE sequencial)."""
        if not src_root.exists():
            return 0

        pairs: list[tuple[Path, Path]] = []
        dirs_to_make: set[Path] = set()
        for src_file in src_root.rglob("*"):
            if not src_file.is_file():
                continue
            rel = src_file.relative_to(src_root)
            dst_file = dst_root / rel
            dirs_to_make.add(dst_file.parent)
            pairs.append((src_file, dst_file))

        for d in sorted(dirs_to_make):
            d.mkdir(parents=True, exist_ok=True)

        def _copy_one(pair: tuple[Path, Path]) -> None:
            shutil.copy2(pair[0], pair[1])

        with ThreadPoolExecutor(max_workers=workers) as pool:
            list(tqdm(pool.map(_copy_one, pairs), total=len(pairs), desc="Sincronizando p/ Drive", leave=False))
        return len(pairs)


    def run_segmentation(
        splits_dir: Path,
        out_dir: Path,
        yolo_weights: str,
        sam_weights: str,
        conf_thresh: float,
        review_thresh: float,
        margin_ratio: float,
        use_sam: bool,
        limit: int | None,
        batch_size: int = DEFAULT_BATCH_SIZE,
        max_infer_side: int = DEFAULT_MAX_INFER_SIDE,
        io_workers: int = DEFAULT_IO_WORKERS,
        use_local_staging: bool = True,
    ) -> None:
        import torch
        from ultralytics import SAM, YOLO

        has_cuda = torch.cuda.is_available()
        device = 0 if has_cuda else "cpu"
        use_half = bool(has_cuda)

        if has_cuda:
            gpu_name = torch.cuda.get_device_name(0)
            print(f"GPU detectada: {gpu_name} | FP16 (half): ATIVADO | Batch: {batch_size}")
        else:
            print("=" * 72)
            print("[AVISO CRÍTICO] Nenhuma GPU CUDA detectada — rodando em CPU!")
            print("No Colab vá em: Ambiente de execução -> Alterar tipo -> GPU (T4 / L4 / A100).")
            print("Em CPU o processo pode levar horas; em GPU leva poucos minutos.")
            print("=" * 72)

        print(f"Carregando detector open-vocabulary: {yolo_weights}")
        detector = YOLO(yolo_weights)
        detector.set_classes(FISH_PROMPTS)

        sam_model = None
        if use_sam:
            print(f"Carregando segmentador SAM 2: {sam_weights}")
            sam_model = SAM(sam_weights)
        else:
            print("Modo rápido (--no-sam): recorte direto pelo YOLO-World sem máscaras SAM 2.")

        out_dir.mkdir(parents=True, exist_ok=True)

        # Usa SSD local do Colab quando a saída está no Google Drive para eliminar lentidão do FUSE
        is_drive_out = "/content/drive/" in str(out_dir).replace("\\", "/")
        staging_root = Path("/content/_fishid_seg_staging") if (use_local_staging and is_drive_out and Path("/content").is_dir()) else out_dir
        if staging_root != out_dir:
            staging_root.mkdir(parents=True, exist_ok=True)
            print(f"Staging SSD local ativado: {staging_root} -> sincronização paralela para {out_dir}")

        review_rows: list[dict] = []
        splits_processed = 0
        t_global = time.perf_counter()

        with ThreadPoolExecutor(max_workers=io_workers) as io_pool:
            for split_name in ("train", "validation", "test"):
                csv_path = splits_dir / f"split_{split_name}.csv"
                if not csv_path.is_file():
                    print(f"[AVISO] Split não encontrado, pulando: {csv_path}")
                    continue
                splits_processed += 1

                df = pd.read_csv(csv_path)
                if limit is not None:
                    df = df.head(limit).copy()

                # Pré-cria todas as subpastas de espécies UMA única vez (zero mkdir no loop!)
                unique_species = sorted({str(s).replace("/", "_").replace(" ", "_") for s in df["species"].unique()})
                for safe_sp in unique_species:
                    (staging_root / "crops" / split_name / safe_sp).mkdir(parents=True, exist_ok=True)
                    if use_sam:
                        (staging_root / "masks" / split_name / safe_sp).mkdir(parents=True, exist_ok=True)

                # Mapa rápido de arquivos já existentes no destino final (retomada sem stat repetido)
                existing_crops: set[str] = set()
                existing_masks: set[str] = set()
                final_crop_split = out_dir / "crops" / split_name
                final_mask_split = out_dir / "masks" / split_name
                if final_crop_split.exists():
                    for root_d, _, files in os.walk(final_crop_split):
                        sp_dir = Path(root_d).name
                        for fname in files:
                            existing_crops.add(f"{sp_dir}/{fname}")
                if use_sam and final_mask_split.exists():
                    for root_d, _, files in os.walk(final_mask_split):
                        sp_dir = Path(root_d).name
                        for fname in files:
                            existing_masks.add(f"{sp_dir}/{fname}")

                new_rows: list[dict] = []
                pending_items: list[dict] = []

                for row in df.itertuples(index=False):
                    src_path = Path(str(row.path))
                    species = str(row.species)
                    label = int(row.label)
                    safe_species = species.replace("/", "_").replace(" ", "_")

                    crop_rel = f"{safe_species}/{src_path.stem}.jpg"
                    mask_rel = f"{safe_species}/{src_path.stem}.png"
                    final_crop = out_dir / "crops" / split_name / crop_rel
                    final_mask = out_dir / "masks" / split_name / mask_rel

                    if crop_rel in existing_crops and (not use_sam or mask_rel in existing_masks):
                        new_rows.append(
                            {
                                "path": str(final_crop),
                                "original_path": str(src_path),
                                "mask_path": str(final_mask) if mask_rel in existing_masks else "",
                                "species": species,
                                "label": label,
                                "status": "cached",
                            }
                        )
                    else:
                        pending_items.append(
                            {
                                "src_path": src_path,
                                "species": species,
                                "label": label,
                                "stage_crop": staging_root / "crops" / split_name / crop_rel,
                                "stage_mask": staging_root / "masks" / split_name / mask_rel,
                                "final_crop": final_crop,
                                "final_mask": final_mask,
                            }
                        )

                print(
                    f"\nSplit '{split_name}': {len(df)} total | {len(new_rows)} já prontos (cache) | {len(pending_items)} a processar"
                )

                pbar = tqdm(total=len(pending_items), desc=f"Split {split_name}", unit="img")
                pending_writes = []

                for start in range(0, len(pending_items), batch_size):
                    chunk = pending_items[start : start + batch_size]

                    # 1) Leitura + pré-redimensionamento em paralelo
                    read_jobs = [(i, it["src_path"], max_infer_side) for i, it in enumerate(chunk)]
                    loaded = list(io_pool.map(_read_image_task, read_jobs))

                    valid_entries: list[tuple[dict, np.ndarray, np.ndarray, float]] = []
                    for idx_local, img_bgr, infer_bgr, scale in loaded:
                        item = chunk[idx_local]
                        if img_bgr is None or infer_bgr is None:
                            review_rows.append(
                                {
                                    "split": split_name,
                                    "path": str(item["src_path"]),
                                    "species": item["species"],
                                    "motivo": "falha_leitura_imagem",
                                    "confianca": 0.0,
                                    "area_mascara_pct": 0.0,
                                }
                            )
                            pbar.update(1)
                        else:
                            valid_entries.append((item, img_bgr, infer_bgr, scale))

                    if not valid_entries:
                        continue

                    # 2) YOLO-World EM LOTE (batch na GPU com FP16!)
                    infer_batch = [entry[2] for entry in valid_entries]
                    det_results = detector.predict(
                        infer_batch,
                        conf=conf_thresh,
                        imgsz=640,
                        device=device,
                        half=use_half,
                        verbose=False,
                    )

                    for (item, img_bgr, infer_bgr, scale), det_res in zip(valid_entries, det_results):
                        img_h, img_w = img_bgr.shape[:2]
                        inf_h, inf_w = infer_bgr.shape[:2]
                        inf_area = float(inf_w * inf_h)
                        boxes = det_res.boxes

                        # Passada 2 (resgate em alta resolução 960px / FP32 / conf 0.025) para casos difíceis
                        if boxes is None or len(boxes) == 0:
                            rescue_res = detector.predict(
                                infer_bgr,
                                conf=min(0.025, conf_thresh),
                                imgsz=960,
                                device=device,
                                verbose=False,
                            )[0]
                            boxes = rescue_res.boxes

                        if boxes is None or len(boxes) == 0:
                            pending_writes.append(
                                io_pool.submit(
                                    _write_image_task,
                                    (item["stage_crop"], img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 92]),
                                )
                            )
                            review_rows.append(
                                {
                                    "split": split_name,
                                    "path": str(item["src_path"]),
                                    "species": item["species"],
                                    "motivo": "nenhum_peixe_detectado",
                                    "confianca": 0.0,
                                    "area_mascara_pct": 100.0,
                                }
                            )
                            new_rows.append(
                                {
                                    "path": str(item["final_crop"]),
                                    "original_path": str(item["src_path"]),
                                    "mask_path": "",
                                    "species": item["species"],
                                    "label": item["label"],
                                    "status": "fallback_full",
                                }
                            )
                            pbar.update(1)
                            continue

                        boxes_xyxy = boxes.xyxy.cpu().numpy()
                        confs = boxes.conf.cpu().numpy()
                        best_box_inf, best_conf, best_area_frac = _select_best_box(boxes_xyxy, confs, inf_area)

                        # Rejeita micro-detecções (< 2,5% da foto, ex.: placa/desenho de peixe na parede)
                        if best_area_frac < 0.025:
                            pending_writes.append(
                                io_pool.submit(
                                    _write_image_task,
                                    (item["stage_crop"], img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 92]),
                                )
                            )
                            review_rows.append(
                                {
                                    "split": split_name,
                                    "path": str(item["src_path"]),
                                    "species": item["species"],
                                    "motivo": "deteccao_muito_pequena_fallback",
                                    "confianca": round(best_conf, 4),
                                    "area_mascara_pct": round(best_area_frac * 100.0, 2),
                                }
                            )
                            new_rows.append(
                                {
                                    "path": str(item["final_crop"]),
                                    "original_path": str(item["src_path"]),
                                    "mask_path": "",
                                    "species": item["species"],
                                    "label": item["label"],
                                    "status": "fallback_full",
                                }
                            )
                            pbar.update(1)
                            continue

                        mask_area_pct = 0.0
                        mask_saved = False
                        final_box_inf = best_box_inf

                        # 3) SAM 2 na imagem redimensionada (até 4x mais rápido que em 12MP!)
                        if sam_model is not None:
                            sam_res = sam_model.predict(
                                infer_bgr,
                                bboxes=[best_box_inf.tolist()],
                                device=device,
                                verbose=False,
                            )[0]
                            if sam_res.masks is not None and len(sam_res.masks.data) > 0:
                                mask_raw = sam_res.masks.data[0].cpu().numpy()
                                mask_u8 = (mask_raw > 0.5).astype(np.uint8) * 255
                                if mask_u8.shape != (inf_h, inf_w):
                                    mask_u8 = cv2.resize(mask_u8, (inf_w, inf_h), interpolation=cv2.INTER_NEAREST)
                                pending_writes.append(
                                    io_pool.submit(
                                        _write_image_task,
                                        (item["stage_mask"], mask_u8, [int(cv2.IMWRITE_PNG_COMPRESSION), 1]),
                                    )
                                )
                                mask_saved = True
                                mask_area_pct = round(100.0 * float(np.count_nonzero(mask_u8)) / max(inf_area, 1.0), 2)
                                tight = _mask_to_clean_bbox(mask_u8)
                                if tight is not None:
                                    # UNIÃO da caixa YOLO + máscara SAM 2: preserva bico/rostro fino e nadadeiras longas
                                    final_box_inf = np.array(
                                        [
                                            min(float(best_box_inf[0]), float(tight[0])),
                                            min(float(best_box_inf[1]), float(tight[1])),
                                            max(float(best_box_inf[2]), float(tight[2])),
                                            max(float(best_box_inf[3]), float(tight[3])),
                                        ],
                                        dtype=np.float32,
                                    )

                        # Reprojeta a caixa para a resolução ORIGINAL e recorta sem perda de qualidade
                        inv_scale = 1.0 / scale
                        x1, y1, x2, y2 = _clamp_box_with_margin(
                            float(final_box_inf[0]) * inv_scale,
                            float(final_box_inf[1]) * inv_scale,
                            float(final_box_inf[2]) * inv_scale,
                            float(final_box_inf[3]) * inv_scale,
                            img_w,
                            img_h,
                            margin_ratio=margin_ratio,
                        )
                        crop_bgr = img_bgr[y1:y2, x1:x2]
                        pending_writes.append(
                            io_pool.submit(
                                _write_image_task,
                                (item["stage_crop"], crop_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 92]),
                            )
                        )

                        motivos = []
                        if best_conf < review_thresh:
                            motivos.append("baixa_confianca")
                        if len(boxes_xyxy) > 2:
                            motivos.append("multiplos_peixes")
                        if use_sam and mask_saved and (mask_area_pct < 3.0 or mask_area_pct > 92.0):
                            motivos.append("area_mascara_atipica")

                        if motivos:
                            review_rows.append(
                                {
                                    "split": split_name,
                                    "path": str(item["src_path"]),
                                    "crop_path": str(item["final_crop"]),
                                    "species": item["species"],
                                    "motivo": ",".join(motivos),
                                    "confianca": round(best_conf, 4),
                                    "area_mascara_pct": mask_area_pct,
                                }
                            )

                        new_rows.append(
                            {
                                "path": str(item["final_crop"]),
                                "original_path": str(item["src_path"]),
                                "mask_path": str(item["final_mask"]) if mask_saved else "",
                                "species": item["species"],
                                "label": item["label"],
                                "status": "ok",
                            }
                        )
                        pbar.update(1)

                    # Evita acumular memória RAM com centenas de gravações pendentes
                    if len(pending_writes) >= batch_size * 4:
                        for fut in pending_writes:
                            fut.result()
                        pending_writes.clear()

                for fut in pending_writes:
                    fut.result()
                pending_writes.clear()
                pbar.close()

                # Sincroniza o split concluído do SSD local para o Google Drive em paralelo
                if staging_root != out_dir and len(pending_items) > 0:
                    copied_crops = _sync_directory_parallel(
                        staging_root / "crops" / split_name,
                        out_dir / "crops" / split_name,
                        workers=io_workers,
                    )
                    copied_masks = (
                        _sync_directory_parallel(
                            staging_root / "masks" / split_name,
                            out_dir / "masks" / split_name,
                            workers=io_workers,
                        )
                        if use_sam
                        else 0
                    )
                    print(f"Sincronizado para o Drive ({split_name}): {copied_crops} crops, {copied_masks} masks")

                out_csv = out_dir / f"split_{split_name}_cropped.csv"
                pd.DataFrame(new_rows).to_csv(out_csv, index=False)
                print(f"Split salvo: {out_csv} ({len(new_rows)} imagens)")

        if splits_processed == 0:
            raise FileNotFoundError(
                f"Nenhum arquivo split_train.csv / split_validation.csv / split_test.csv encontrado em: {splits_dir}\n"
                "Passe o caminho correto do seu Google Drive em --splits-dir (ou edite DEFAULT_SPLITS_DIR) "
                "e salve a saída no Drive em --out-dir (ou DEFAULT_OUT_DIR)."
            )

        review_csv = out_dir / "revisao_manual.csv"
        pd.DataFrame(review_rows).to_csv(review_csv, index=False)
        elapsed_min = (time.perf_counter() - t_global) / 60.0
        print(f"\nConcluído em {elapsed_min:.1f} min! Casos para revisão manual: {len(review_rows)} -> {review_csv}")

    _splits_dir = Path(DEFAULT_SPLITS_DIR)
    _out_dir = Path(DEFAULT_OUT_DIR)

    # Limpa com segurança apenas se a pasta ainda tiver só o teste antigo (<= 100 crops)
    if LIMPAR_TESTE_ANTIGO and (_out_dir / "crops").exists():
        _n_prev = sum(1 for _ in (_out_dir / "crops").rglob("*.jpg"))
        if _n_prev <= 100:
            print(f"Limpando teste antigo ({_n_prev} crops) para reprocessar com a correção do rostro/anti-placa...")
            shutil.rmtree(_out_dir, ignore_errors=True)
            shutil.rmtree(Path("/content/_fishid_seg_staging"), ignore_errors=True)
        else:
            print(f"Cache completo/parcial detectado ({_n_prev} crops) — retomando de onde parou sem apagar nada!")

    run_segmentation(
        splits_dir=_splits_dir,
        out_dir=_out_dir,
        yolo_weights="yolov8l-worldv2.pt",
        sam_weights="sam2.1_t.pt",
        conf_thresh=0.08,
        review_thresh=0.25,
        margin_ratio=0.15,
        use_sam=True,
        limit=LIMITE_SEGMENTACAO,
        batch_size=DEFAULT_BATCH_SIZE,
        max_infer_side=DEFAULT_MAX_INFER_SIDE,
        io_workers=DEFAULT_IO_WORKERS,
        use_local_staging=True,
    )
else:
    print("⏭️ Seção 0 pulada (RODAR_SEGMENTACAO = False). Usando recortes já salvos no Drive.")


In [ ]:
# =============================================================================
# 1. Configurações principais do experimento
# =============================================================================
MODO = "treinar"                    # "treinar" (Warm-Start a partir dos ~85%), "analisar" (SÓ avalia) ou "resultados"
USAR_DATASET_SEGMENTADO = True      # True = usa crops YOLO-World + SAM 2 | False = fotos originais
PERMITIR_SUBSET_PEQUENO = False     # Proteção: bloqueia treino acidental se split_train_cropped.csv ainda tiver só 20 fotos do teste
COPIAR_PARA_SSD_LOCAL = True        # Copia imagens para o SSD do Colab antes do treino (muito mais rápido!)
MODELO = "convnext_tiny_segmentado" if USAR_DATASET_SEGMENTADO else "convnext_tiny_otimizado"
SEED = 42

# Reaproveitamento automático do seu melhor modelo já treinado (ex.: o de ~80% de ontem!)
USAR_MELHOR_MODELO_SALVO = True     # True = parte dos pesos do melhor modelo.keras já salvo no Drive (não começa do zero!)
MODELO_PRETREINADO_PATH = ""        # Opcional: caminho exato de um modelo.keras. Se vazio (""), acha o campeão automaticamente!
PULAR_FASE_A_COM_PRETREINADO = True # Já estando em ~80%, pula a Fase A e vai direto para o fine-tuning suave (Fase B)

# Caminhos no Google Drive
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/modelos_peixes/fonte_verdade_experimentos"
BASELINE_RUN_NAME = "run_20261011_165821"
DRIVE_SEG_DIR = "/content/drive/MyDrive/modelos_peixes/dataset_segmentado"

# Resolução e preservação morfológica
IMG_SIZE = (320, 320)          # Na A100 você pode testar até (384, 384)
PRESERVAR_PROPORCAO = True     # True usa resize_with_pad (crucial para peixes alongados)
BATCH_SIZE = 32                # Use 16 se subir para (384, 384) em GPU T4
USAR_MIXED_PRECISION = True    # Acelera bastante na A100/L4/T4

# Estratégia de treinamento em 2 fases
EPOCHS_HEAD = 12
EPOCHS_FINE = 35
LR_HEAD = 5e-4
LR_FINE = 5e-5                 # Pico após warmup quando começa do zero (ImageNet)
LR_FINE_WARM = 2e-5            # LR mais suave quando já parte do seu modelo campeão de ~80%
WEIGHT_DECAY = 1e-2            # Padrão do paper original do ConvNeXt
DROPOUT_RATE = 0.35
ULTIMAS_CAMADAS_FINE_TUNE = 95 # Libera estágios 2 e 3 (ou None para liberar tudo)

# Regularização e avaliação
USAR_MIXUP = True
MIXUP_ALPHA = 0.20
USAR_TTA = True                # Média entre foto original e espelhada no teste
GERAR_GRADCAM = True
N_GRADCAM = 6
ESPECIE_GRADCAM = None         # Ex.: "Tetrapturus pfluegeri"


## 2. Conectar ao Drive, localizar seu melhor modelo já salvo e carregar o baseline


In [ ]:
from pathlib import Path
from google.colab import drive
from IPython.display import Image, display
import json
import pandas as pd

if MODO not in {"resultados", "analisar", "treinar"}:
    raise ValueError('Escolha "resultados", "analisar" ou "treinar".')

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

RESULTS_ROOT = Path(DRIVE_RESULTS_ROOT)
SEG_DIR = Path(DRIVE_SEG_DIR)
origem = RESULTS_ROOT / BASELINE_RUN_NAME
resumo = json.loads((origem / "resumo_experimento.json").read_text(encoding="utf-8"))
resultados_salvos = pd.DataFrame(resumo["results"])
baseline_convnext = resultados_salvos.loc[resultados_salvos["model"].eq("convnext_tiny")].reset_index(drop=True)

def localizar_melhor_modelo_salvo(results_root: Path, caminho_manual: str = ""):
    if caminho_manual.strip():
        p = Path(caminho_manual.strip())
        if not p.is_file():
            raise FileNotFoundError(f"MODELO_PRETREINADO_PATH não encontrado: {p}")
        return p, None
    candidatos = []
    for mk in sorted(results_root.glob("run_*/**/modelo.keras")):
        # Ignora runs de teste rápido com menos de 100 imagens no split_test.csv
        run_root = mk.parent.parent
        test_csv = run_root / "split_test.csv"
        if test_csv.is_file():
            try:
                if len(pd.read_csv(test_csv)) < 100:
                    continue
            except Exception:
                pass
        acc = -1.0
        met_csv = mk.parent / "metricas_teste.csv"
        if met_csv.is_file():
            try:
                df_m = pd.read_csv(met_csv)
                if "accuracy" in df_m.columns and not df_m.empty:
                    # Exclui eventual linha de baseline inserida na tabela comparativa
                    mask_self = ~df_m["variante"].astype(str).str.lower().str.startswith("baseline") if "variante" in df_m.columns else pd.Series([True] * len(df_m))
                    vals = df_m.loc[mask_self, "accuracy"].dropna()
                    if not vals.empty:
                        acc = float(vals.max())
            except Exception:
                pass
        candidatos.append((acc, mk.stat().st_mtime, mk))
    if not candidatos:
        return None, None
    candidatos.sort(key=lambda t: (t[0], t[1]), reverse=True)
    best_acc, _, best_path = candidatos[0]
    return best_path, (best_acc if best_acc >= 0 else None)

MELHOR_MODELO_PATH, MELHOR_MODELO_ACC = (
    localizar_melhor_modelo_salvo(RESULTS_ROOT, MODELO_PRETREINADO_PATH)
    if (USAR_MELHOR_MODELO_SALVO or MODO == "analisar")
    else (None, None)
)

print(f"Baseline original ({BASELINE_RUN_NAME}):")
display(baseline_convnext)
print(f"Modo execução: {MODO.upper()} | Dataset: {'SEGMENTADO (YOLO-World + SAM 2)' if USAR_DATASET_SEGMENTADO else 'ORIGINAL (fotos inteiras)'}")
if MELHOR_MODELO_PATH is not None:
    acc_str = f" (accuracy teste: {MELHOR_MODELO_ACC:.2%})" if MELHOR_MODELO_ACC is not None else ""
    print(f"✅ Melhor modelo salvo encontrado{acc_str}:\n   {MELHOR_MODELO_PATH}")
else:
    print("ℹ️ Nenhum modelo.keras anterior válido encontrado — o treino começará do ImageNet.")
cm_baseline = origem / "convnext_tiny" / "confusion_matrix.png"
if cm_baseline.exists():
    display(Image(filename=str(cm_baseline), width=720))


## 3. Preparar splits (originais ou segmentados), cache em SSD local e pipeline `tf.data`


In [ ]:
if MODO != "resultados":
    from concurrent.futures import ThreadPoolExecutor
    from datetime import datetime
    import math
    import shutil
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, top_k_accuracy_score

    keras.utils.set_random_seed(SEED)
    gpus = tf.config.list_physical_devices("GPU")
    if USAR_MIXED_PRECISION and gpus:
        keras.mixed_precision.set_global_policy("mixed_float16")
    else:
        keras.mixed_precision.set_global_policy("float32")

    USE_CLASS_WEIGHT = resumo.get("use_class_weight", True)
    CLASS_WEIGHT_POWER = resumo.get("class_weight_power", 0.5)
    CLASS_WEIGHT_MAX = resumo.get("class_weight_max", 3.0)
    LABEL_SMOOTHING = resumo.get("label_smoothing", 0.1)
    AUTOTUNE = tf.data.AUTOTUNE

    indices = json.loads((origem / "class_indices.json").read_text(encoding="utf-8"))
    classes = sorted(indices, key=indices.get)
    num_classes = len(classes)
    if classes != resumo["classes"] or [indices[c] for c in classes] != list(range(num_classes)):
        raise ValueError("A ordem de classes diverge da execução original.")

    ausentes_log = []
    def carregar_split(nome):
        if USAR_DATASET_SEGMENTADO:
            csv_path = SEG_DIR / f"split_{nome}_cropped.csv"
            if not csv_path.is_file():
                raise FileNotFoundError(
                    f"Arquivo segmentado não encontrado: {csv_path}. "
                    "Aguarde o término do segmentar_peixes_auto.py ou mude USAR_DATASET_SEGMENTADO = False."
                )
        else:
            csv_path = origem / f"split_{nome}.csv"

        tabela = pd.read_csv(csv_path)
        if not {"path", "species", "label"} <= set(tabela.columns):
            raise ValueError(f"Colunas do split {nome} incompletas.")
        if tabela.empty or tabela["path"].isna().any() or tabela["path"].duplicated().any():
            raise ValueError(f"O split {nome} contém caminhos inválidos, vazios ou repetidos.")
        if not tabela["label"].eq(tabela["species"].map(indices)).all():
            raise ValueError(f"Rótulos inconsistentes no split {nome}.")
        # Verifica se os arquivos existem no Drive (evita NotFoundError no meio da época)
        existe_mask = tabela["path"].astype(str).map(lambda p: Path(p).is_file())
        n_faltando = int((~existe_mask).sum())
        if n_faltando > 0:
            print(f"[AVISO] Split '{nome}': {n_faltando} arquivo(s) não encontrado(s) no Drive e removido(s) com segurança.")
            for p_miss, sp_miss in zip(tabela.loc[~existe_mask, "path"], tabela.loc[~existe_mask, "species"]):
                ausentes_log.append({"split": nome, "species": sp_miss, "path": p_miss})
            tabela = tabela.loc[existe_mask].reset_index(drop=True)
        if tabela.empty:
            raise FileNotFoundError(f"Nenhum arquivo do split '{nome}' foi encontrado no Drive. Verifique o caminho do dataset.")
        if USAR_DATASET_SEGMENTADO and "status" in tabela.columns:
            status_counts = tabela["status"].value_counts().to_dict()
            print(f"Split '{nome}' (segmentado): {len(tabela)} imagens | status={status_counts}")
        return tabela

    train_df = carregar_split("train")
    val_df = carregar_split("validation")
    test_df = carregar_split("test")

    if USAR_DATASET_SEGMENTADO and not PERMITIR_SUBSET_PEQUENO and len(train_df) < 1000:
        msg_subset = (
            f"Seu split_train_cropped.csv só tem {len(train_df)} imagens (teste rápido DEFAULT_LIMIT = 20)! "
            "Rode segmentar_peixes_auto.py com DEFAULT_LIMIT = None para gerar todas as ~4.354 fotos de treino, 933 de validação e 933 de teste."
        )
        if MODO == "treinar":
            raise RuntimeError(f"[TRAVA DE SEGURANÇA] {msg_subset}")
        print(f"[AVISO] {msg_subset}")

    paths = [set(tabela["path"]) for tabela in (train_df, val_df, test_df)]
    if sum(map(len, paths)) != len(set.union(*paths)):
        raise ValueError("Existe sobreposição de caminhos entre os splits.")

    # Espelha os arquivos para o SSD local do Colab para eliminar lentidão do Google Drive FUSE durante o treino
    SSD_CACHE_DIR = Path("/content/_fishid_train_ssd")
    def preparar_paths_rapidos(df, split_nome):
        if not (COPIAR_PARA_SSD_LOCAL and Path("/content").is_dir()):
            df["fast_path"] = df["path"]
            return df
        sub_dir = SSD_CACHE_DIR / ("seg" if USAR_DATASET_SEGMENTADO else "orig") / split_nome
        sub_dir.mkdir(parents=True, exist_ok=True)
        jobs = []
        fast_paths = []
        for idx_r, src_str in enumerate(df["path"].astype(str)):
            src = Path(src_str)
            dst = sub_dir / f"{idx_r:05d}_{src.name}"
            fast_paths.append(str(dst))
            if not dst.is_file():
                jobs.append((src, dst))
        if jobs:
            with ThreadPoolExecutor(max_workers=12) as pool:
                list(pool.map(lambda pair: shutil.copy2(pair[0], pair[1]), jobs))
        df["fast_path"] = fast_paths
        return df

    if COPIAR_PARA_SSD_LOCAL:
        print("Copiando imagens para SSD local do Colab (acelera em até 5x cada época)...")
    train_df = preparar_paths_rapidos(train_df, "train")
    val_df = preparar_paths_rapidos(val_df, "validation")
    test_df = preparar_paths_rapidos(test_df, "test")

    print(f"Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")
    print("TensorFlow:", tf.__version__, "| GPUs:", gpus, "| Policy:", keras.mixed_precision.global_policy().name)
    print("\nDistribuição por espécie no treino:")
    display(train_df["species"].value_counts().rename("n_treino").to_frame())

    # Augmentation realista para fotos de campo/embarcação (entrada em [0, 1])
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10, fill_mode="reflect"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect"),
        layers.RandomContrast(0.18),
        layers.RandomBrightness(0.15, value_range=(0.0, 1.0)),
    ], name="field_augmentation")

    def ler_imagem(path):
        img = tf.image.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        img = tf.image.convert_image_dtype(img, tf.float32)  # [0, 1]
        if PRESERVAR_PROPORCAO:
            img = tf.image.resize_with_pad(img, IMG_SIZE[0], IMG_SIZE[1], antialias=True)
        else:
            img = tf.image.resize(img, IMG_SIZE, antialias=True)
        img.set_shape((*IMG_SIZE, 3))
        return img

    def aplicar_mixup(images, labels, alpha=MIXUP_ALPHA):
        images = tf.cast(images, tf.float32)
        labels = tf.cast(labels, tf.float32)
        batch_size = tf.shape(images)[0]
        g1 = tf.random.gamma([batch_size], alpha, dtype=tf.float32)
        g2 = tf.random.gamma([batch_size], alpha, dtype=tf.float32)
        lam = g1 / (g1 + g2 + 1e-7)
        lam = tf.maximum(lam, 1.0 - lam)
        idx = tf.random.shuffle(tf.range(batch_size))
        lam_img = tf.reshape(lam, [batch_size, 1, 1, 1])
        lam_lbl = tf.reshape(lam, [batch_size, 1])
        mixed_images = images * lam_img + tf.gather(images, idx) * (1.0 - lam_img)
        mixed_labels = labels * lam_lbl + tf.gather(labels, idx) * (1.0 - lam_lbl)
        return mixed_images, mixed_labels

    def make_dataset(split_df, training=False, use_mixup=False):
        file_paths = split_df["fast_path"].astype(str).to_numpy()
        labels = split_df["label"].astype("int32").to_numpy()
        ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
        if training:
            ds = ds.shuffle(buffer_size=len(split_df), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(
            lambda p, y: (ler_imagem(p), tf.one_hot(y, num_classes)),
            num_parallel_calls=AUTOTUNE,
        )
        if training:
            # Proteção extra contra JPEGs corrompidos ou oscilações do Google Drive FUSE
            ds = ds.ignore_errors(log_warning=True)
        ds = ds.batch(BATCH_SIZE)
        if training:
            ds = ds.map(
                lambda x, y: (tf.cast(tf.clip_by_value(data_augmentation(x, training=True), 0.0, 1.0), tf.float32), tf.cast(y, tf.float32)),
                num_parallel_calls=AUTOTUNE,
            )
            if use_mixup:
                ds = ds.map(aplicar_mixup, num_parallel_calls=AUTOTUNE)
        return ds.prefetch(AUTOTUNE)

    if MODO == "treinar":
        train_ds = make_dataset(train_df, training=True, use_mixup=USAR_MIXUP)
        val_ds = make_dataset(val_df, training=False, use_mixup=False)
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        RUN_DIR = RESULTS_ROOT / f"run_{RUN_ID}_{MODELO}"
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        for nome, split in (("train", train_df), ("validation", val_df), ("test", test_df)):
            split.drop(columns=["fast_path"], errors="ignore").to_csv(RUN_DIR / f"split_{nome}.csv", index=False)
        (RUN_DIR / "class_indices.json").write_text(json.dumps(indices, ensure_ascii=False, indent=2), encoding="utf-8")
        # Exporta a lista exata de classes para o FishID Studio
        (RUN_DIR / "class_names_fishid.json").write_text(json.dumps(classes, ensure_ascii=False, indent=2), encoding="utf-8")
        if ausentes_log:
            pd.DataFrame(ausentes_log).to_csv(RUN_DIR / "arquivos_ausentes.csv", index=False)

        counts = train_df["label"].value_counts().sort_index()
        weights = len(train_df) / (num_classes * counts)
        weights = weights.pow(CLASS_WEIGHT_POWER).clip(upper=CLASS_WEIGHT_MAX)
        # Nota: com MixUp ativo, rótulos viram soft-labels; desativamos class_weight se MixUp estiver ligado
        class_weight = {int(k): float(v) for k, v in weights.items()} if (USE_CLASS_WEIGHT and not USAR_MIXUP) else None
        saida = RUN_DIR / MODELO
    else:
        saida = origem / "analise_individual" / MODELO

    saida.mkdir(parents=True, exist_ok=True)


## 3.1 Inspeção Visual da Segmentação (Original | Máscara SAM 2 | Recorte) — Figura para o Paper


In [ ]:
if MODO != "resultados" and USAR_DATASET_SEGMENTADO:
    from PIL import Image as PILImage

    N_EXEMPLOS_MASCARAS = 10  # 1 exemplo de cada uma das 10 espécies (pode aumentar se quiser)
    amostras_ok = test_df.copy()
    if "mask_path" in amostras_ok.columns and "original_path" in amostras_ok.columns:
        amostras_ok = amostras_ok[
            amostras_ok["mask_path"].fillna("").astype(str).map(lambda p: bool(p) and Path(p).is_file())
            & amostras_ok["original_path"].fillna("").astype(str).map(lambda p: bool(p) and Path(p).is_file())
        ]
    # Seleciona 1 exemplo por espécie (cobrindo todas as 10 espécies) e completa até N_EXEMPLOS_MASCARAS
    por_esp = amostras_ok.groupby("species", group_keys=False).head(1)
    if len(por_esp) < N_EXEMPLOS_MASCARAS:
        extras = amostras_ok.drop(index=por_esp.index).head(N_EXEMPLOS_MASCARAS - len(por_esp))
        amostras_fig = pd.concat([por_esp, extras], ignore_index=True)
    else:
        amostras_fig = por_esp.head(N_EXEMPLOS_MASCARAS).reset_index(drop=True)

    if not amostras_fig.empty:
        fig, axs = plt.subplots(len(amostras_fig), 3, figsize=(13.5, 3.2 * len(amostras_fig)), squeeze=False)
        for i, row in enumerate(amostras_fig.itertuples(index=False)):
            orig_im = PILImage.open(row.original_path).convert("RGB")
            mask_im = PILImage.open(row.mask_path).convert("L")
            crop_im = PILImage.open(row.fast_path).convert("RGB")
            mask_arr = np.asarray(mask_im)
            area_pct = 100.0 * float(np.count_nonzero(mask_arr > 127)) / max(mask_arr.size, 1)

            axs[i, 0].imshow(orig_im)
            axs[i, 0].set_title(f"{i+1}. Original · {row.species}", fontsize=10, fontweight="bold")
            axs[i, 1].imshow(mask_im, cmap="gray")
            axs[i, 1].set_title(f"Máscara SAM 2.1 (área: {area_pct:.1f}%)", fontsize=10)
            axs[i, 2].imshow(crop_im)
            axs[i, 2].set_title("Recorte (+15% margem) → ConvNeXt-Tiny", fontsize=10, color="#0A385C")
            for ax in axs[i]:
                ax.axis("off")

        fig.suptitle(f"Pipeline de Auto-Segmentação (YOLO-World + SAM 2.1) — {len(amostras_fig)} Espécies · UNIVALI · LEMA", fontsize=12.5, fontweight="bold")
        fig.tight_layout(rect=[0, 0, 1, 0.99])
        fig.savefig(saida / "comparativo_segmentacao.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("Figura comparativa salva em:", saida / "comparativo_segmentacao.png")


## 4. Arquitetura limpa (sem `Lambda`, pronta para FishID Studio) e Fine-Tuning com CosineDecay


In [ ]:
if MODO != "resultados":
    def construir_modelo(weights="imagenet"):
        inputs = keras.Input(shape=(*IMG_SIZE, 3), name="image")
        # Camada nativa Rescaling(255.0) substitui Lambda -> 100% compatível com keras.models.load_model()
        # No FishID Studio (app.py), mantenha NORMALIZE_DIV_255 = True para enviar [0, 1] aqui.
        x = layers.Rescaling(255.0, name="to_255_scale")(inputs)
        base = tf.keras.applications.ConvNeXtTiny(
            include_top=False,
            include_preprocessing=True,
            weights=weights,
            input_tensor=x,
        )
        base.trainable = False
        x = layers.GlobalAveragePooling2D(name="gap")(base.output)
        x = layers.LayerNormalization(epsilon=1e-6, name="head_ln")(x)
        x = layers.Dropout(DROPOUT_RATE, name="dropout")(x)
        # dtype='float32' na saída garante estabilidade numérica com mixed_float16
        outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="predictions")(x)
        model = keras.Model(inputs, outputs, name="ConvNeXtTiny_FishID")
        return model, base

if MODO == "treinar":
    steps_per_epoch = math.ceil(len(train_df) / BATCH_SIZE)

    def build_cosine_schedule(peak_lr, epochs, warmup_epochs=2):
        total_steps = max(1, epochs * steps_per_epoch)
        warmup_steps = min(total_steps // 4, warmup_epochs * steps_per_epoch)
        return keras.optimizers.schedules.CosineDecay(
            initial_learning_rate=peak_lr * 0.1,
            decay_steps=max(1, total_steps - warmup_steps),
            warmup_target=peak_lr,
            warmup_steps=warmup_steps,
            alpha=0.02,
        )

    def compile_model(model, lr_schedule):
        model.compile(
            optimizer=keras.optimizers.AdamW(
                learning_rate=lr_schedule,
                weight_decay=WEIGHT_DECAY,
                clipnorm=1.0,
            ),
            loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
            metrics=[
                "accuracy",
                keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_accuracy"),
            ],
        )
        return model

    def make_callbacks(stage_dir, patience=10):
        stage_dir.mkdir(parents=True, exist_ok=True)
        return [
            keras.callbacks.ModelCheckpoint(
                filepath=str(stage_dir / "best.keras"),
                monitor="val_accuracy",
                mode="max",
                save_best_only=True,
                verbose=1,
            ),
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                mode="max",
                patience=patience,
                restore_best_weights=True,
                verbose=1,
            ),
            keras.callbacks.CSVLogger(str(stage_dir / "history.csv")),
        ]

    usando_warm_start = False
    if USAR_MELHOR_MODELO_SALVO and MELHOR_MODELO_PATH is not None:
        print(f"\n🚀 Warm-Start ativado! Carregando pesos do seu melhor modelo salvo:\n   {MELHOR_MODELO_PATH}")
        modelo, base_modelo = construir_modelo(weights=None)
        try:
            modelo.load_weights(str(MELHOR_MODELO_PATH))
            usando_warm_start = True
            print("✅ Pesos carregados com sucesso! Partindo direto do seu modelo campeão (~80%).")
        except Exception as exc_w:
            print(f"[AVISO] load_weights falhou ({exc_w}); carregando modelo completo via load_model...")
            modelo = keras.models.load_model(str(MELHOR_MODELO_PATH), compile=False)
            base_modelo = modelo
            usando_warm_start = True
    else:
        modelo, base_modelo = construir_modelo(weights="imagenet")

    # ---------------- FASE A: Treino da cabeça (pulada no Warm-Start!) ----------------
    if usando_warm_start and PULAR_FASE_A_COM_PRETREINADO:
        print("\n⏭️ Pulando Fase A (modelo já parte dos pesos treinados de ~80%) -> indo direto para o Fine-Tuning suave!")
    else:
        print("\n===== FASE A: Cabeça classificadora (backbone congelado) =====")
        head_dir = RUN_DIR / f"{MODELO}_head"
        compile_model(modelo, build_cosine_schedule(LR_HEAD, EPOCHS_HEAD, warmup_epochs=1))
        history_head = modelo.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS_HEAD,
            callbacks=make_callbacks(head_dir, patience=5),
            class_weight=class_weight,
        )
        # Garante partir exatamente dos melhores pesos da Fase A
        best_head_path = head_dir / "best.keras"
        if best_head_path.exists():
            modelo.load_weights(str(best_head_path))

    # ---------------- FASE B: Fine-tuning real ----------------
    lr_fase_b = LR_FINE_WARM if usando_warm_start else LR_FINE
    print(f"\n===== FASE B: Fine-tuning do ConvNeXt-Tiny (pico LR={lr_fase_b}) =====")
    base_modelo.trainable = True
    if ULTIMAS_CAMADAS_FINE_TUNE is not None:
        for layer in base_modelo.layers[:-ULTIMAS_CAMADAS_FINE_TUNE]:
            layer.trainable = False
        print(f"Liberadas as últimas {ULTIMAS_CAMADAS_FINE_TUNE} de {len(base_modelo.layers)} camadas.")
    else:
        print(f"Backbone inteiro liberado ({len(base_modelo.layers)} camadas).")

    fine_dir = RUN_DIR / f"{MODELO}_finetune"
    compile_model(modelo, build_cosine_schedule(lr_fase_b, EPOCHS_FINE, warmup_epochs=2 if usando_warm_start else 3))
    history_fine = modelo.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_FINE,
        callbacks=make_callbacks(fine_dir, patience=10),
        class_weight=class_weight,
    )

    best_fine_path = fine_dir / "best.keras"
    if best_fine_path.exists():
        modelo.load_weights(str(best_fine_path))

    # Salva modelo final pronto para copiar como backend/models/modelo.keras no FishID Studio
    modelo_final_path = saida / "modelo.keras"
    modelo.save(modelo_final_path)
    print("\nModelo pronto para o FishID Studio salvo em:", modelo_final_path)


## 5. Avaliação no conjunto de teste (sem TTA vs. com TTA) + Matriz de Confusão


In [ ]:
if MODO != "resultados":
    if MODO == "analisar":
        _model_path = globals().get("MELHOR_MODELO_PATH")
        if _model_path is None:
            # Busca automaticamente o melhor modelo.keras salvo no Drive (ignorando testes <100 fotos)
            _cands = []
            for _mk in sorted(RESULTS_ROOT.glob("run_*/**/modelo.keras")):
                _tcsv = _mk.parent.parent / "split_test.csv"
                if _tcsv.is_file():
                    try:
                        if len(pd.read_csv(_tcsv)) < 100:
                            continue
                    except Exception:
                        pass
                _acc = -1.0
                _mcsv = _mk.parent / "metricas_teste.csv"
                if _mcsv.is_file():
                    try:
                        _dfm = pd.read_csv(_mcsv)
                        if "accuracy" in _dfm.columns and not _dfm.empty:
                            _mask = ~_dfm["variante"].astype(str).str.lower().str.startswith("baseline") if "variante" in _dfm.columns else pd.Series([True] * len(_dfm))
                            _vals = _dfm.loc[_mask, "accuracy"].dropna()
                            if not _vals.empty:
                                _acc = float(_vals.max())
                    except Exception:
                        pass
                _cands.append((_acc, _mk.stat().st_mtime, _mk))
            if not _cands:
                raise FileNotFoundError(f"Nenhum modelo.keras encontrado em {RESULTS_ROOT}")
            _cands.sort(key=lambda t: (t[0], t[1]), reverse=True)
            _model_path = _cands[0][2]
        print(f"Carregando modelo salvo para análise (sem treinar): {_model_path}")
        modelo = keras.models.load_model(str(_model_path), compile=False)

    def inferir_dataset(model, flip_horizontal=False):
        ds = tf.data.Dataset.from_tensor_slices(test_df["fast_path"].astype(str).to_numpy())
        ds = ds.map(ler_imagem, num_parallel_calls=AUTOTUNE)
        if flip_horizontal:
            ds = ds.map(tf.image.flip_left_right, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
        return model.predict(ds, verbose=1)

    probs_base = inferir_dataset(modelo, flip_horizontal=False)
    if USAR_TTA:
        probs_flip = inferir_dataset(modelo, flip_horizontal=True)
        probabilidades = 0.5 * (probs_base + probs_flip)
    else:
        probabilidades = probs_base

    reais = test_df["label"].to_numpy(dtype=int)
    etiquetas = list(range(num_classes))

    def avaliar(probs, tag):
        preds = probs.argmax(axis=1)
        rep = classification_report(reais, preds, labels=etiquetas, target_names=classes, output_dict=True, zero_division=0)
        return {
            "variante": tag,
            "accuracy": accuracy_score(reais, preds),
            "top3_accuracy": top_k_accuracy_score(reais, probs, k=3, labels=etiquetas),
            "macro_f1": rep["macro avg"]["f1-score"],
            "weighted_f1": rep["weighted avg"]["f1-score"],
        }, rep

    linhas_metricas = []
    if not baseline_convnext.empty:
        b_row = baseline_convnext.iloc[0]
        linhas_metricas.append({
            "variante": f"Baseline original ({BASELINE_RUN_NAME})",
            "accuracy": float(b_row.get("test_accuracy", b_row.get("accuracy", float("nan")))),
            "top3_accuracy": float(b_row.get("test_top3_accuracy", b_row.get("top3_accuracy", float("nan")))),
            "macro_f1": float(b_row.get("test_macro_f1", b_row.get("macro_f1", float("nan")))),
            "weighted_f1": float(b_row.get("test_weighted_f1", b_row.get("weighted_f1", float("nan")))),
        })
    m_base, rep_base = avaliar(probs_base, f"{MODELO} (sem TTA)")
    linhas_metricas.append(m_base)
    if USAR_TTA:
        m_tta, rep_final = avaliar(probabilidades, f"{MODELO} (com TTA)")
        linhas_metricas.append(m_tta)
    else:
        rep_final = rep_base

    tabela_metricas = pd.DataFrame(linhas_metricas)
    tabela_metricas.to_csv(saida / "metricas_teste.csv", index=False)
    print(f"\n===== Comparativo de Métricas no Teste ({len(test_df)} imagens) =====")
    display(tabela_metricas)

    por_especie = pd.DataFrame(rep_final).T
    por_especie.to_csv(saida / "classification_report.csv")
    print("\nDesempenho por espécie (ordenado do menor para o maior recall):")
    display(por_especie.loc[classes].sort_values("recall"))

    previstos = probabilidades.argmax(axis=1)
    predicoes = test_df.drop(columns=["fast_path"], errors="ignore").copy()
    predicoes["fast_path"] = test_df["fast_path"].values
    predicoes["predita"] = [classes[i] for i in previstos]
    predicoes["confianca"] = probabilidades.max(axis=1)
    predicoes["acertou"] = previstos == reais
    predicoes.drop(columns=["fast_path"], errors="ignore").to_csv(saida / "predicoes.csv", index=False)

    cm = confusion_matrix(reais, previstos, labels=etiquetas)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    cm_df.to_csv(saida / "confusion_matrix.csv")

    fig, ax = plt.subplots(figsize=(11, 8))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="crest", ax=ax)
    ax.set(title=f"Matriz de confusão — {MODELO}", xlabel="Predito", ylabel="Real")
    ax.tick_params(axis="x", labelrotation=55)
    fig.tight_layout()
    fig.savefig(saida / "confusion_matrix.png", dpi=150)
    plt.show()
    plt.close(fig)
    print("Resultados salvos em:", saida)
    print("\nLista CLASS_NAMES para o app.py do FishID Studio:")
    print(json.dumps(classes, ensure_ascii=False, indent=2))


## 6. Diagnóstico Didático com Grad-CAM: Onde o modelo erra nos peixes de bico?

### Como interpretar os 3 painéis (guia rápido de pesquisa)
Para cada erro de alta confiança (especialmente em **peixes de bico** — família *Istiophoridae*, como *Tetrapturus*, *Makaira*, *Kajikia* e *Istiophorus* — ou no par mais confundido da matriz), esta célula compara lado a lado:
1. **Foto original + Top-3 probabilidades:** mostra se a espécie correta ficou em 2º lugar por margem estreita (falta de detalhe fino) ou se o modelo errou com convicção.
2. **Grad-CAM da Classe Prevista (ERRO):** regiões quentes (vermelho/amarelo) indicam *por que* a rede escolheu a espécie errada. Se o calor estiver no **barco, mão do pescador, água ou convés**, há viés de fundo (*shortcut learning* — sinal forte para usar recorte/segmentação).
3. **Grad-CAM da Classe Real (CORRETA):** mostra onde a rede encontraria evidência para a espécie verdadeira. Compare com o painel 2: se os dois mapas olham para o **rostro (bico), nadadeira dorsal ou peitoral**, o erro é de taxonomia fina (justifica maior resolução `384×384` e `resize_with_pad`).


In [ ]:
if MODO != "resultados" and GERAR_GRADCAM:
    # -------------------------------------------------------------------------
    # 6.1 Ranking automático dos pares de espécies mais confundidos
    # -------------------------------------------------------------------------
    pares = []
    for i_real, esp_real in enumerate(classes):
        for i_pred, esp_pred in enumerate(classes):
            if i_real != i_pred and cm[i_real, i_pred] > 0:
                pares.append({
                    "especie_real": esp_real,
                    "predita_errada": esp_pred,
                    "erros": int(cm[i_real, i_pred]),
                    "pct_da_especie_real": round(100.0 * cm[i_real, i_pred] / max(1, int(cm[i_real].sum())), 1),
                })
    pares_df = pd.DataFrame(pares).sort_values("erros", ascending=False).reset_index(drop=True)
    pares_df.to_csv(saida / "pares_mais_confundidos.csv", index=False)
    print("===== Top 10 pares de espécies mais confundidos no teste =====")
    display(pares_df.head(10))

    # -------------------------------------------------------------------------
    # 6.2 Extrator Grad-CAM compatível com ConvNeXt-Tiny + mixed_float16
    # -------------------------------------------------------------------------
    # Seleciona a última camada espacial 4D do modelo (saída do stage 3 do ConvNeXt)
    camada_alvo = next(
        l for l in reversed(modelo.layers)
        if hasattr(l, "output") and len(l.output.shape) == 4
    )
    idx_alvo = modelo.layers.index(camada_alvo)
    extrator_feat = keras.Model(modelo.inputs, camada_alvo.output, name="gradcam_extractor")
    camadas_cabeca = modelo.layers[idx_alvo + 1:]
    print(f"Camada espacial usada no Grad-CAM: {camada_alvo.name} | shape={camada_alvo.output.shape}")

    def calcular_gradcam(img_tensor, classe_idx):
        x_batch = tf.expand_dims(img_tensor, axis=0)
        feat = tf.cast(extrator_feat(x_batch, training=False), tf.float32)
        with tf.GradientTape() as tape:
            tape.watch(feat)
            z = feat
            for l in camadas_cabeca[:-1]:
                z = l(z, training=False)
            dense = camadas_cabeca[-1]
            # Usa o logit pré-softmax para evitar saturação do gradiente
            logits = tf.linalg.matmul(tf.cast(z, tf.float32), tf.cast(dense.kernel, tf.float32))
            if dense.use_bias:
                logits = tf.nn.bias_add(logits, tf.cast(dense.bias, tf.float32))
            score = logits[:, int(classe_idx)]
        grads = tape.gradient(score, feat)
        if grads is None:
            return np.zeros(feat.shape[1:3], dtype=np.float32), False
        pesos = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
        mapa = tf.nn.relu(tf.reduce_sum(feat * pesos, axis=-1))[0]
        pico = float(tf.reduce_max(mapa))
        if pico <= 1e-8:
            return np.zeros(mapa.shape, dtype=np.float32), False
        return (mapa / pico).numpy(), True

    def descrever_foco(mapa):
        h_m, w_m = mapa.shape
        yy, xx = np.unravel_index(int(np.argmax(mapa)), mapa.shape)
        vert = ["superior", "central", "inferior"][min(2, int(3 * yy / max(h_m, 1)))]
        horiz = ["esquerda", "centro", "direita"][min(2, int(3 * xx / max(w_m, 1)))]
        # Verifica se a energia está muito concentrada nas bordas (forte indício de viés de fundo)
        borda = np.ones_like(mapa, dtype=bool)
        borda[1:-1, 1:-1] = False
        pct_borda = float(mapa[borda].sum() / (mapa.sum() + 1e-8))
        alerta = " (atenção nas bordas/fundo!)" if pct_borda > 0.55 else ""
        return f"região {vert}-{horiz}{alerta}", round(pct_borda * 100.0, 1)

    # -------------------------------------------------------------------------
    # 6.3 Seleção inteligente de erros (prioriza peixes de bico se existirem)
    # -------------------------------------------------------------------------
    erros_df = predicoes[~predicoes["acertou"]].copy()
    if ESPECIE_GRADCAM is not None:
        erros_filtrados = erros_df[erros_df["species"] == ESPECIE_GRADCAM]
    else:
        # Detecta automaticamente peixes de bico (Istiophoridae / Xiphias) nos nomes científicos
        termos_bico = ("tetrapturus", "makaira", "kajikia", "istiophorus", "xiphias", "marlin", "agulh", "espad")
        mask_bico = erros_df["species"].str.lower().apply(lambda s: any(t in s for t in termos_bico))
        erros_filtrados = erros_df[mask_bico] if mask_bico.any() else erros_df

    # Prioriza os erros em que o modelo errou com MAIOR confiança (mais informativos!)
    erros_filtrados = erros_filtrados.sort_values("confianca", ascending=False)
    selecionados = list(erros_filtrados.index[:N_GRADCAM])

    if not selecionados:
        print("Nenhum erro encontrado para o filtro selecionado!")
    else:
        print(f"\nGerando comparativo Grad-CAM para os {len(selecionados)} erros mais confiantes...")
        fig, axs = plt.subplots(len(selecionados), 3, figsize=(15, 4.4 * len(selecionados)), squeeze=False)
        diagnosticos = []

        for row_idx, sample_idx in enumerate(selecionados):
            row = predicoes.loc[sample_idx]
            img_t = ler_imagem(row["fast_path"])
            img_np = np.clip(img_t.numpy(), 0.0, 1.0)
            probs_i = probabilidades[sample_idx]
            top3_idx = np.argsort(probs_i)[::-1][:3]
            top3_txt = "\n".join([f"{k+1}º {classes[c]} ({probs_i[c]:.1%})" for k, c in enumerate(top3_idx)])

            c_pred = int(previstos[sample_idx])
            c_real = int(reais[sample_idx])
            mapa_pred, ok_pred = calcular_gradcam(img_t, c_pred)
            mapa_real, ok_real = calcular_gradcam(img_t, c_real)

            # Painel 1: Foto original + Top-3
            axs[row_idx, 0].imshow(img_np)
            axs[row_idx, 0].set_title(
                f"REAL: {classes[c_real]}\nERROU PARA: {classes[c_pred]} ({probs_i[c_pred]:.1%})\n{top3_txt}",
                fontsize=9.5,
                loc="left",
            )

            # Painel 2: Grad-CAM da classe ERRADA (prevista)
            foco_pred, borda_pred = descrever_foco(mapa_pred) if ok_pred else ("sem ativação", 0.0)
            axs[row_idx, 1].imshow(img_np)
            if ok_pred:
                up_pred = tf.image.resize(mapa_pred[..., None], IMG_SIZE).numpy()[..., 0]
                axs[row_idx, 1].imshow(up_pred, cmap="jet", alpha=0.42, vmin=0, vmax=1)
            axs[row_idx, 1].set_title(
                f"ONDE OLHOU PARA ERRAR (Predita)\n{classes[c_pred]} · {foco_pred}",
                fontsize=9.5,
                color="#B91C1C",
            )

            # Painel 3: Grad-CAM da classe CERTA (real)
            foco_real, borda_real = descrever_foco(mapa_real) if ok_real else ("sem ativação", 0.0)
            axs[row_idx, 2].imshow(img_np)
            if ok_real:
                up_real = tf.image.resize(mapa_real[..., None], IMG_SIZE).numpy()[..., 0]
                axs[row_idx, 2].imshow(up_real, cmap="jet", alpha=0.42, vmin=0, vmax=1)
            axs[row_idx, 2].set_title(
                f"ONDE DEVERIA OLHAR (Real: {probs_i[c_real]:.1%})\n{classes[c_real]} · {foco_real}",
                fontsize=9.5,
                color="#0F766E",
            )

            for ax in axs[row_idx]:
                ax.axis("off")

            diagnosticos.append({
                "arquivo": Path(row["path"]).name,
                "real": classes[c_real],
                "predita_errada": classes[c_pred],
                "prob_predita": round(float(probs_i[c_pred]), 4),
                "prob_real": round(float(probs_i[c_real]), 4),
                "real_no_top3": bool(c_real in top3_idx),
                "foco_erro": foco_pred,
                "energia_borda_erro_pct": borda_pred,
                "foco_real": foco_real,
            })

        fig.suptitle(
            "Diagnóstico Grad-CAM de Erros — Predita Errada (vermelho) vs. Classe Real (verde-petróleo)",
            fontsize=13,
            fontweight="bold",
        )
        fig.tight_layout(rect=[0, 0, 1, 0.98])
        fig.savefig(saida / "gradcam_erros_comparativo.png", dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        diag_df = pd.DataFrame(diagnosticos)
        diag_df.to_csv(saida / "gradcam_diagnostico_erros.csv", index=False)
        print("\nResumo tabular dos erros analisados (salvo em gradcam_diagnostico_erros.csv):")
        display(diag_df)
